# Watermark ablations

Original Llama watermark-design ablations followed by the corrected Qwen watermark-strength sweep.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))
import numpy as np
from util.filereader import load_path_file

## 1. ID & K_p Ablation

In [2]:
base_path = ["results", "_ablations", "watermark", "1_id_kp_ablation"]
sub_experiment_names = ["unique_ids", "unique_kps"]
sacct_results = load_path_file(["results", "_ablations", "watermark", "1_id_kp_ablation"], "sacct_results.json")["jobs"]
for i, sub_experiment_name in enumerate(sub_experiment_names):
    experiment_path = base_path + [sub_experiment_name]
    scores = load_path_file(experiment_path, "scores.json")
    keys = load_path_file(experiment_path, "keys.json")
    if sub_experiment_name == "unique_ids":
        print("Unique IDs")
        num_correct = 0
        for sample, expected_id in zip(scores, keys["ids"]):
            predicted_id = np.argmax([x[0] for x in sample]) + 1 # because 0 indexed but id is 1-indexed
            if predicted_id == expected_id:
                num_correct += 1
        accuracy = num_correct / len(scores)
        print("N_samples:", len(scores))
        print("Accuracy:", accuracy)
    elif sub_experiment_name == "unique_kps":
        print("Unique K_ps")
        num_correct = 0
        for sample, expected_kp in zip(scores, keys["k_ps"]):
            predicted_kp = np.argmax(sample) + 1
            if predicted_kp == expected_kp:
                num_correct += 1
        accuracy = num_correct / len(scores)
        print("N_samples:", len(scores))
        print("Accuracy:", accuracy)

    latencies = load_path_file(experiment_path, "latency.json")
    latency_watermark = round(latencies["watermark"], 2)
    latency_verify = round(latencies["verify"], 2)
    print("Watermark time:", latency_watermark, "s")
    print("Verification time:", latency_verify, "s")
    job_results = sacct_results[i]
    print("Total runtime:", job_results["elapsed"])
    print("Max Memory used:", round(job_results["max_rss_kb"]/(1024*1024), 2), "GB")
    print("")

Unique IDs
N_samples: 200
Accuracy: 0.88
Watermark time: 1326.13 s
Verification time: 2682.8 s
Total runtime: 01:08:09
Max Memory used: 104.63 GB

Unique K_ps
N_samples: 200
Accuracy: 0.875
Watermark time: 1232.03 s
Verification time: 18.7 s
Total runtime: 00:24:32
Max Memory used: 8.08 GB



## 2. Watermark Function Ablation

In [3]:
base_path = ["results", "_ablations", "watermark", "2_watermarkfn_ablation"]
sub_experiment_names = ["fourier", "square"]
for sub_experiment_name in sub_experiment_names:
    experiment_path = base_path + [sub_experiment_name]
    scores = load_path_file(experiment_path, "scores.json")
    keys = load_path_file(experiment_path, "keys.json")

    print(sub_experiment_name)
    num_correct = 0
    for sample, expected_kp in zip(scores, keys["k_ps"]):
        predicted_kp = np.argmax(sample) + 1
        if predicted_kp == expected_kp:
            num_correct += 1
    accuracy = num_correct / len(scores)
    print("N_samples:", len(scores))
    print("Accuracy:", accuracy)

    latencies = load_path_file(experiment_path, "latency.json")
    latency_watermark = round(latencies["watermark"], 2)
    latency_verify = round(latencies["verify"], 2)
    print("Watermark time:", latency_watermark, "s")
    print("Verification time:", latency_verify, "s")
    print("")

fourier
N_samples: 1000
Accuracy: 0.8
Watermark time: 6078.73 s
Verification time: 133.15 s

square
N_samples: 10
Accuracy: 0.9
Watermark time: 58.24 s
Verification time: 1.82 s



## 3. N-gram Ablation

In [4]:
base_path = ["results", "_ablations", "watermark", "3_ngram_ablation"]
sub_experiment_names = ["one", "two", "three", "four", "five", "six"]

# reported using $ sacct -j <jobid> --format=JobID,MaxRSS,MaxVMSize,Elapsed,CPUTime
sacct_results = load_path_file(base_path, "sacct_results.json")["jobs"]

for i, sub_experiment_name in enumerate(sub_experiment_names):
    experiment_path = base_path + [sub_experiment_name]
    scores = load_path_file(experiment_path, "scores.json")
    keys = load_path_file(experiment_path, "keys.json")

    print(sub_experiment_name)
    num_correct = 0
    for sample, expected_kp in zip(scores, keys["k_ps"]):
        predicted_kp = np.argmax(sample) + 1
        if predicted_kp == expected_kp:
            num_correct += 1
    accuracy = num_correct / len(scores)
    print("N_samples:", len(scores))
    print("Accuracy:", accuracy)

    latencies = load_path_file(experiment_path, "latency.json")
    latency_watermark = round(latencies["watermark"], 2)
    latency_verify = round(latencies["verify"], 2)
    print("Watermark time:", latency_watermark, "s")
    print("Verification time:", latency_verify, "s")
    job_results = sacct_results[i]
    print("Total runtime:", job_results["elapsed"])
    print("Max Memory used:", round(job_results["max_rss_kb"]/(1024*1024), 2), "GB")
    print("")

one
N_samples: 1000
Accuracy: 0.002
Watermark time: 5621.32 s
Verification time: 102.34 s
Total runtime: 01:36:59
Max Memory used: 33.04 GB



two
N_samples: 1000
Accuracy: 0.818
Watermark time: 6334.37 s
Verification time: 137.67 s
Total runtime: 01:49:10
Max Memory used: 39.48 GB



three
N_samples: 1000
Accuracy: 0.844
Watermark time: 6459.53 s
Verification time: 288.01 s
Total runtime: 01:53:45
Max Memory used: 27.52 GB



four
N_samples: 1000
Accuracy: 0.85
Watermark time: 6571.78 s
Verification time: 394.55 s
Total runtime: 01:58:02
Max Memory used: 23.16 GB



five
N_samples: 1000
Accuracy: 0.842
Watermark time: 6540.26 s
Verification time: 432.79 s
Total runtime: 01:57:31
Max Memory used: 46.6 GB



six
N_samples: 1000
Accuracy: 0.838
Watermark time: 6480.5 s
Verification time: 447.75 s
Total runtime: 01:56:46
Max Memory used: 50.08 GB



## 4. Kappa Ablation

In [5]:
base_path = ["results", "_ablations", "watermark", "4_kappa_ablation"]
sub_experiment_names = ["two", "four", "six", "eight"]

for i, sub_experiment_name in enumerate(sub_experiment_names):
    experiment_path = base_path + [sub_experiment_name]
    scores = load_path_file(experiment_path, "scores.json")
    sts_scores = load_path_file(experiment_path, "sts_scores.json")
    keys = load_path_file(experiment_path, "keys.json")

    print(sub_experiment_name)
    num_correct = 0
    for sample, expected_kp in zip(scores, keys["k_ps"]):
        predicted_kp = np.argmax(sample) + 1
        if predicted_kp == expected_kp:
            num_correct += 1
    sts_avg = 0
    for score in sts_scores:
        sts_avg += score
    sts_avg = sts_avg / len(sts_scores)
    accuracy = num_correct / len(scores)
    print("N_samples:", len(scores))
    print("Accuracy:", accuracy)

    latencies = load_path_file(experiment_path, "latency.json")
    latency_watermark = round(latencies["watermark"], 2)
    latency_verify = round(latencies["verify"], 2)
    print("Watermark time:", latency_watermark, "s")
    print("Verification time:", latency_verify, "s")
    print("Avg STS score: ", sts_avg)
    print("")

two
N_samples: 1000
Accuracy: 0.816
Watermark time: 6415.45 s
Verification time: 134.82 s
Avg STS score:  0.9405899963676929



four
N_samples: 1000
Accuracy: 0.945
Watermark time: 6336.34 s
Verification time: 132.15 s
Avg STS score:  0.9319784786738455



six
N_samples: 1000
Accuracy: 0.951
Watermark time: 6588.89 s
Verification time: 131.87 s
Avg STS score:  0.9241607461422682



eight
N_samples: 1000
Accuracy: 0.949
Watermark time: 7489.05 s
Verification time: 145.97 s
Avg STS score:  0.9177190342694521



## Paper-ready ablation tables

The Qwen table includes every final length-corrected κ condition. Semantic similarity is retained as a column and reported as `NA` where it has not been computed or saved yet.

In [6]:
import json
import pandas as pd
from IPython.display import display

repo_root = Path.cwd().resolve()
if not (repo_root / "results").is_dir():
    repo_root = Path.cwd().resolve().parents[1]
ablation_root = repo_root / "results/_ablations/watermark"

qwen_rows = []
qwen_root = ablation_root / "qwen_sampling_source_lengthfix"
for summary_path in sorted(qwen_root.glob("*/summary.json")):
    summary = json.loads(summary_path.read_text())
    sts_path = summary_path.parent / "sts_scores.json"
    sts = json.loads(sts_path.read_text()) if sts_path.is_file() else []
    qwen_rows.append({
        **summary,
        "semantic_similarity": float(np.mean(sts)) if sts else np.nan,
        "condition": summary_path.parent.name,
    })
qwen_ablation_table = pd.DataFrame(qwen_rows).sort_values("kappa")
qwen_columns = [
    "kappa", "temperature", "top_p", "top_k", "n_texts", "n_candidates",
    "top1_accuracy", "top10_accuracy", "mean_reciprocal_rank", "mean_rank",
    "mean_correct_score", "mean_correct_vs_best_wrong_margin",
    "semantic_similarity", "max_new_tokens_ratio",
]
qwen_ablation_table = qwen_ablation_table.reindex(columns=qwen_columns)
display(qwen_ablation_table.round(4).fillna("NA"))

table_dir = repo_root / "src/eval/tables/ablations"
table_dir.mkdir(parents=True, exist_ok=True)
qwen_ablation_table.to_csv(table_dir / "qwen_watermark_strength.csv", index=False)
print(qwen_ablation_table.to_latex(
    index=False, float_format=lambda x: f"{x:.4f}", na_rep="--"
))

,kappa,temperature,top_p,top_k,n_texts,n_candidates,top1_accuracy,top10_accuracy,mean_reciprocal_rank,mean_rank,mean_correct_score,mean_correct_vs_best_wrong_margin,semantic_similarity,max_new_tokens_ratio
2,2.0,1.0,1.0,0,100,100,0.57,0.89,0.6796,4.75,0.1578,0.0048,NA,1.5
4,3.0,1.0,1.0,0,100,100,0.90,0.99,0.9305,1.66,0.3015,0.1346,NA,1.5
3,3.5,1.0,1.0,0,100,100,0.95,1.00,0.9683,1.10,0.4182,0.2347,NA,1.5
5,4.0,1.0,1.0,0,100,100,0.99,1.00,0.9950,1.01,0.5270,0.3412,NA,1.5
6,6.0,1.0,1.0,0,100,100,0.98,1.00,0.9831,1.11,0.8441,0.5486,NA,1.5
7,8.0,1.0,1.0,0,100,100,0.99,1.00,0.9925,1.03,0.8964,0.5946,NA,1.5
0,10.0,1.0,1.0,0,100,100,0.98,1.00,0.9845,1.07,0.9254,0.5828,NA,1.5
1,12.0,1.0,1.0,0,100,100,0.98,1.00,0.9853,1.06,0.9371,0.5690,NA,1.5


\begin{tabular}{rrrrrrrrrrrrrr}
\toprule
kappa & temperature & top_p & top_k & n_texts & n_candidates & top1_accuracy & top10_accuracy & mean_reciprocal_rank & mean_rank & mean_correct_score & mean_correct_vs_best_wrong_margin & semantic_similarity & max_new_tokens_ratio \\
\midrule
2.0000 & 1.0000 & 1.0000 & 0 & 100 & 100 & 0.5700 & 0.8900 & 0.6796 & 4.7500 & 0.1578 & 0.0048 & -- & 1.5000 \\
3.0000 & 1.0000 & 1.0000 & 0 & 100 & 100 & 0.9000 & 0.9900 & 0.9305 & 1.6600 & 0.3015 & 0.1346 & -- & 1.5000 \\
3.5000 & 1.0000 & 1.0000 & 0 & 100 & 100 & 0.9500 & 1.0000 & 0.9683 & 1.1000 & 0.4182 & 0.2347 & -- & 1.5000 \\
4.0000 & 1.0000 & 1.0000 & 0 & 100 & 100 & 0.9900 & 1.0000 & 0.9950 & 1.0100 & 0.5270 & 0.3412 & -- & 1.5000 \\
6.0000 & 1.0000 & 1.0000 & 0 & 100 & 100 & 0.9800 & 1.0000 & 0.9831 & 1.1100 & 0.8441 & 0.5486 & -- & 1.5000 \\
8.0000 & 1.0000 & 1.0000 & 0 & 100 & 100 & 0.9900 & 1.0000 & 0.9925 & 1.0300 & 0.8964 & 0.5946 & -- & 1.5000 \\
10.0000 & 1.0000 & 1.0000 & 0 & 100 & 100 & 